# Sim 6: Opus-Teacher PDA Distillation + Perspective Sweep

**Teacher:** Claude Opus 4.6 (strongest available)
**Student:** Google Gemma-2-9B (dense, text-only, 4bit QLoRA)
**Platform:** Kaggle GPU T4 (16GB)

**Experiments:**
1. Base vs CoT vs PDA-2 vs PDA-3 vs PDA-4 vs PDA-5 distillation
2. Best perspective count identified
3. All evaluated on GSM8K (200 questions)

**Key:** Different family from Qwen (Sim 5 student). Shows PDA distillation generalizes across model families. Gemma-2 (text-only) chosen over Gemma-3 due to Gemma-3 multimodal wrapper overhead on T4.

**Upload training data as Kaggle Dataset:**
- `opus_cot_gsm8k.jsonl`
- `opus_pda2_gsm8k.jsonl`
- `opus_pda3_gsm8k.jsonl`
- `opus_pda4_gsm8k.jsonl`
- `opus_pda5_gsm8k.jsonl`

**Setup:** Kaggle > Notebooks > New > Accelerator: GPU T4 x2 (nutzt 1). HF-Token fuer Gemma-Zugriff in Secrets hinterlegen (Gemma ist gated).

In [ ]:
%%capture
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
import os, sys, json, re, random, torch

# Use unsloth's pre-quantized 4bit Gemma-2-9b (fast download, no gating)
MODEL_ID = "unsloth/gemma-2-9b-bnb-4bit"

# === PLATFORM DETECTION ===
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if ON_KAGGLE:
    PLATFORM = "kaggle"
    DATA_DIR = "/kaggle/input/datasets/t0bybr/simulation6"
    OUTPUT_DIR = "/kaggle/working"
    MODEL_DIR = MODEL_ID  # download from HF (unsloth is not gated)
elif ON_COLAB:
    PLATFORM = "colab"
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/models"
    MODEL_DIR = MODEL_ID
    OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/models"
else:
    PLATFORM = "local"
    DATA_DIR = "."
    MODEL_DIR = MODEL_ID
    OUTPUT_DIR = "."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Platform: {PLATFORM}, Device: {DEVICE}")
print(f"Data: {DATA_DIR}")
print(f"Model: {MODEL_DIR}")
print(f"Output: {OUTPUT_DIR}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

## 1. Load Training Data

In [ ]:
def load_data(filename, reasoning_key="reasoning"):
    examples = []
    path = os.path.join(DATA_DIR, filename)
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            if d.get("correct", False):
                rkey = reasoning_key if reasoning_key in d else "pda_reasoning"
                examples.append({
                    "question": d["question"],
                    "reasoning": d.get(rkey, d.get("reasoning", "")),
                })
    return examples

# Load all variants
data = {}
data["cot"] = load_data("opus_cot_gsm8k.jsonl", "reasoning")
data["pda2"] = load_data("opus_pda2_gsm8k.jsonl", "pda_reasoning")
data["pda3"] = load_data("opus_pda3_gsm8k.jsonl", "pda_reasoning")
data["pda4"] = load_data("opus_pda4_gsm8k.jsonl", "pda_reasoning")
data["pda5"] = load_data("opus_pda5_gsm8k.jsonl", "pda_reasoning")

for k, v in data.items():
    print(f"{k}: {len(v)} correct examples")

## 2. Training Loop — One Adapter Per Variant

In [ ]:
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import Dataset

max_seq_length = 2048

def load_fresh_model():
    """Load Gemma-2-9B pre-quantized 4bit via unsloth."""
    print(f"Loading model from: {MODEL_DIR}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_DIR,
        max_seq_length=max_seq_length,
        dtype=None,  # auto (fp16 on T4)
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16, lora_alpha=16, lora_dropout=0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    return model, tokenizer

def format_example(example):
    return f"Question: {example['question']}\n\nSolution: {example['reasoning']}"

def make_dataset(examples):
    return Dataset.from_list([
        {"text": format_example(ex)} for ex in examples
    ])

def train_variant(variant_name, examples):
    save_path = os.path.join(OUTPUT_DIR, f"adapter-opus-{variant_name}")

    if os.path.exists(os.path.join(save_path, "adapter_model.safetensors")):
        print(f"\n=== {variant_name}: Already trained, skipping ===")
        return

    print(f"\n{'='*60}")
    print(f"Training {variant_name} adapter on {len(examples)} examples")
    print(f"{'='*60}")

    model, tokenizer = load_fresh_model()
    dataset = make_dataset(examples)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            num_train_epochs=3,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=42,
            output_dir=save_path + "-checkpoints",
            save_strategy="no",
            report_to="none",
        ),
    )

    stats = trainer.train()
    print(f"Loss: {stats.training_loss:.4f}")

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved to {save_path}")

    del model, trainer
    torch.cuda.empty_cache()

print("Training functions ready (unsloth).")

In [ ]:
# Train all variants sequentially
for variant, examples in data.items():
    train_variant(variant, examples)

print("\n=== All adapters trained ===")

## 3. Evaluation — All Variants on GSM8K

In [ ]:
from datasets import load_dataset

N_EVAL = 200
random.seed(42)

def extract_number(text):
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    if m: return float(m.group(1).replace(",", ""))
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    for n in reversed(nums):
        c = n.replace(",", "").strip()
        if c and c != "-":
            try: return float(c)
            except: continue
    return None

def normalize(s):
    if s is None: return None
    s = str(s).strip().replace(" ", "").lower()
    try: return str(float(s))
    except: return s

gsm8k_test = load_dataset("openai/gsm8k", "main", split="test")
test_idx = list(range(len(gsm8k_test)))
random.shuffle(test_idx)
test_idx = test_idx[:N_EVAL]

def evaluate(model, tokenizer, tag):
    correct = total = 0
    for i, idx in enumerate(test_idx):
        item = gsm8k_test[idx]
        m = re.search(r'####\s*(-?[\d,]+\.?\d*)', item["answer"])
        if not m: continue
        gt = float(m.group(1).replace(",", ""))
        
        # Format as plain text (base model, no chat template)
        prompt = f"Question: {item['question']}\n\nSolution:"
        ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            out = model.generate(input_ids=ids, max_new_tokens=256,
                                temperature=0.0, do_sample=False)
        resp = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
        pred = extract_number(resp)
        
        if pred is not None and normalize(str(pred)) == normalize(str(gt)):
            correct += 1
        total += 1
        
        if (i+1) % 50 == 0:
            print(f"  [{tag}] {i+1}/{N_EVAL}: {correct}/{total} ({100*correct/total:.1f}%)")
    
    acc = round(100*correct/total, 1) if total else 0
    print(f"  [{tag}] Final: {correct}/{total} ({acc}%)")
    return {"correct": correct, "total": total, "accuracy": acc}

print(f"Test set: {len(test_idx)} questions")

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

# Load base model once (pre-quantized 4bit via unsloth)
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_DIR,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

# === BASELINE ===
print("=" * 60)
print("BASELINE (Gemma-2-9B, no adapter)")
print("=" * 60)
results = {}
results["base"] = evaluate(base_model, tokenizer, "Base")

# === Each adapter variant ===
for variant in ["cot", "pda2", "pda3", "pda4", "pda5"]:
    adapter_path = os.path.join(OUTPUT_DIR, f"adapter-opus-{variant}")
    if not os.path.exists(adapter_path):
        print(f"\nSkipping {variant} (adapter not found at {adapter_path})")
        continue

    print(f"\n{'='*60}")
    print(f"{variant.upper()} DISTILLED")
    print(f"{'='*60}")

    adapted = PeftModel.from_pretrained(base_model, adapter_path)
    FastLanguageModel.for_inference(adapted)
    results[variant] = evaluate(adapted, tokenizer, variant.upper())

    del adapted
    torch.cuda.empty_cache()

## 4. Results

In [ ]:
print("\n" + "=" * 70)
print("  SIM 6: OPUS-TEACHER PDA DISTILLATION + PERSPECTIVE SWEEP")
print("=" * 70)
print(f"  Teacher: Claude Opus 4.6")
print(f"  Student: Gemma-3-12B-PT (dense, 4bit QLoRA)")
print(f"  Benchmark: GSM8K ({N_EVAL} questions)")
print("=" * 70)

base_acc = results.get("base", {}).get("accuracy", 0)
print(f"\n  {'Variant':<12} {'Accuracy':>10} {'vs Base':>10} {'vs CoT':>10}")
print("  " + "-" * 42)

cot_acc = results.get("cot", {}).get("accuracy", 0)
best_variant = "base"
best_acc = base_acc

for variant in ["base", "cot", "pda2", "pda3", "pda4", "pda5"]:
    if variant not in results:
        continue
    acc = results[variant]["accuracy"]
    vs_base = acc - base_acc
    vs_cot = acc - cot_acc if variant != "base" else 0

    base_str = f"+{vs_base:.1f}pp" if vs_base > 0 else f"{vs_base:.1f}pp" if vs_base < 0 else "---"
    cot_str = f"+{vs_cot:.1f}pp" if vs_cot > 0 else f"{vs_cot:.1f}pp" if vs_cot < 0 else "---"
    marker = " <-- best" if acc > best_acc else ""
    if acc > best_acc:
        best_acc = acc
        best_variant = variant

    print(f"  {variant:<12} {acc:>8.1f}%  {base_str:>10} {cot_str:>10}{marker}")

print("  " + "-" * 42)
print(f"\n  Best variant: {best_variant} ({best_acc}%)")

# Perspective analysis
pda_accs = {k: v["accuracy"] for k, v in results.items() if k.startswith("pda")}
if pda_accs:
    best_pda = max(pda_accs, key=pda_accs.get)
    print(f"  Best PDA: {best_pda} ({pda_accs[best_pda]}%)")
    if cot_acc > 0:
        all_pda_beat_cot = all(v > cot_acc for v in pda_accs.values())
        any_pda_beat_cot = any(v > cot_acc for v in pda_accs.values())
        if all_pda_beat_cot:
            print("  All PDA variants outperform CoT distillation.")
        elif any_pda_beat_cot:
            print("  Some PDA variants outperform CoT distillation.")
        else:
            print("  No PDA variant outperforms CoT distillation.")

# Save
with open(os.path.join(OUTPUT_DIR, "sim6_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\n  Saved to {OUTPUT_DIR}/sim6_results.json")